# Cleaning2
## Inizializzazione ed Import

In [ ]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [ ]:
file_codes = ['UPENN_ROI_MARS']
# 'ADNIMERGE', 'MMSE', 'PTDEMOG', 'ADSP_PHC_BIOMARKER', 'BLCHANGE', 'DXSUM', 
# 'UCSFFSX', 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL', 'UCSDVOL'

In [ ]:
search = client.query_files(
    query={'custom.level' : 'cleaned_01', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


## Operazioni
- Eliminare i parametri con troppe poche righe
- Eliminare soffetti con solo 1 visita
- nuovi metadati (cofattori e fattori)

In [ ]:
# create new support file with the info from the new dataset after cleaning 2
support_file_path = 'ADNI_variables_cleaned1'
support_file = pd.read_excel(support_file_path+'.xlsx')
new_name = 'ADNI_variables_cleaned2'

if os.path.isfile(new_name+'.xlsx'):
    #aggiunge i filecode mancanti e riporta i file_code da riprocessare allo status precedente (variable names)
    update_new_support_file(support_file, new_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_name, rename_column=False)


new_support_file = pd.read_excel(new_name+'.xlsx')
dataCleaner = DataCleaner(support_file=new_support_file)

In [ ]:
for file_name in zip_files.keys():
    print('\n\n ----', file_name)
    
    df = zip_files[file_name]
    df_new = df.copy(deep=True)
    new_support_file = dataCleaner.update_self_support_file(new_support_file)
    # Eliminare i parametri con troppe poche righesia dal DF che dal support file
    if file_name not in []:  #eccezioni
        df_cleaned, file_code, new_support_file = dataCleaner.remove_param_few_subjects(df_new, file_name, prefix='cleaned/single_file')
    # --> funzione che trova i soggetti che hanno solo una visita quindi elimina quelle righe
    if file_code not in ['ADSP_PHC_BIOMARKER', 'PTDEMOG', 'UPENN_ROI_MARS']:         # file solo con 1 visita, o info demog che anche una sola visita basta perchè baseline quindi da unire per ampliare il dataset ma non da usare da solo
        # Eliminare soggetti con solo 1 visita
        df_cleaned= dataCleaner.remove_sub_1visit(df_cleaned)
        # --> funzione che trova i parametri identificati da eliminare  ==> eliminare le colonne dal df

    # funzione che trasforma parametri categorici in dummies
    ref_list = ['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']
    to_dummy_list = [x for x in ref_list if x in df_cleaned.columns]
    if to_dummy_list:
        final_df, bool_var = dataCleaner.classes_to_dummies(df_cleaned, col_list=to_dummy_list) 
    else:
        final_df = df_cleaned
    
    new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_02')

    # aggiunta di righe per i nuovi parametri e rimozione dal support di variabili non più nel df
    new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
    
    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_02', updated_support_file=new_support_file)    
    
    # get info into the new support file
    infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name, prefix='cleaned/single_file')
    
    for key in final_df.keys():
        if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
            new_support_file = infoSupportFile.get_varible_info(key)
    
    # upload the new file
    result = client.upload_dataframe(
        df=final_df,
        object_name=new_file_name,
        prefix='cleaned/single_file/',
        metadata=updated_metadata
    )

save_df(df_to_save=new_support_file, output_path=new_name) 


In [ ]:
updated_metadata